# Research notebook
Run the bootstrap cell first. Review the experiment parameters and data paths before executing the remaining cells. Outputs are intentionally cleared for version control.


In [ ]:
from pathlib import Path
import os
import sys
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
os.chdir(project_root)
sys.path.insert(0, str(project_root / "src"))
Path("runs/notebooks").mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# COMPARE GLASSO STATE GRAPH RESULTS ACROSS MULTIPLE LAGS
# input : saved outputs from lag-specific folders
# output: structured comparison report printed in notebook
# ============================================================

import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

RUNS = {
    "lag_3":  "./glasso_cache_v4_lag3",
    "lag_10": "./glasso_cache_v4_lag10",
    "lag_20": "./glasso_cache_v4_lag20",
}

TOP_N = 10
FLOAT_FMT = "{:.6f}"


# ============================================================
# UTILS
# ============================================================

def print_rule(char="=", width=100):
    print(char * width)


def print_section(title):
    print()
    print_rule("=")
    print(title)
    print_rule("=")


def print_subsection(title):
    print()
    print_rule("-")
    print(title)
    print_rule("-")


def load_json(path):
    if not os.path.exists(path):
        print(f"[WARNING] Missing JSON: {path}")
        return {}
    with open(path, "r") as f:
        return json.load(f)


def load_csv(path):
    if not os.path.exists(path):
        print(f"[WARNING] Missing CSV: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


def fmt_df(df, n=None):
    if df.empty:
        print("[empty]")
        return
    if n is not None:
        df = df.head(n)
    print(df.to_string(index=False))


def safe_get(d, key, default=np.nan):
    return d.get(key, default)


def run_paths(base_dir):
    return {
        "summary_json": os.path.join(
            base_dir, "state_graph_final", "lob_state_glasso_summary.json"
        ),
        "edge_analysis_json": os.path.join(
            base_dir, "state_graph_edge_analysis", "lob_state_edge_analysis_summary.json"
        ),
        "ranking_summary_json": os.path.join(
            base_dir, "state_graph_edge_rankings", "lob_state_edge_rankings_summary.json"
        ),
        "summary_by_lag_type": os.path.join(
            base_dir, "state_graph_edge_analysis", "summary_by_lag_type.csv"
        ),
        "summary_by_level_relation": os.path.join(
            base_dir, "state_graph_edge_analysis", "summary_by_level_relation.csv"
        ),
        "summary_by_feature_pair": os.path.join(
            base_dir, "state_graph_edge_analysis", "summary_by_feature_pair.csv"
        ),
        "summary_by_side_pair": os.path.join(
            base_dir, "state_graph_edge_analysis", "summary_by_side_pair.csv"
        ),
        "top_edges": os.path.join(
            base_dir, "state_graph_edge_rankings", "top_edges_overall.csv"
        ),
        "top_cross_lag": os.path.join(
            base_dir, "state_graph_edge_rankings", "top_edges_cross_lag.csv"
        ),
        "top_intra_lag": os.path.join(
            base_dir, "state_graph_edge_rankings", "top_edges_intra_lag.csv"
        ),
    }


def add_run_col(df, run_name):
    if df.empty:
        return df
    out = df.copy()
    out.insert(0, "run", run_name)
    return out


# ============================================================
# LOAD ALL RUNS
# ============================================================

all_data = {}

for run_name, base_dir in RUNS.items():
    paths = run_paths(base_dir)

    all_data[run_name] = {
        "base_dir": base_dir,
        "paths": paths,
        "summary": load_json(paths["summary_json"]),
        "edge_analysis": load_json(paths["edge_analysis_json"]),
        "ranking_summary": load_json(paths["ranking_summary_json"]),
        "summary_by_lag_type": load_csv(paths["summary_by_lag_type"]),
        "summary_by_level_relation": load_csv(paths["summary_by_level_relation"]),
        "summary_by_feature_pair": load_csv(paths["summary_by_feature_pair"]),
        "summary_by_side_pair": load_csv(paths["summary_by_side_pair"]),
        "top_edges": load_csv(paths["top_edges"]),
        "top_cross_lag": load_csv(paths["top_cross_lag"]),
        "top_intra_lag": load_csv(paths["top_intra_lag"]),
    }


# ============================================================
# 1) GLOBAL COMPARISON TABLE
# ============================================================

rows = []

for run_name, data in all_data.items():
    s = data["summary"]
    e = data["edge_analysis"]

    row = {
        "run": run_name,
        "base_dir": data["base_dir"],

        "best_alpha": safe_get(s, "best_alpha"),
        "fit_time_sec": safe_get(s, "fit_time_sec"),
        "n_samples": safe_get(s, "n_samples"),
        "n_features": safe_get(s, "n_features"),
        "n_possible_edges": safe_get(s, "n_possible_edges"),
        "n_edges": safe_get(s, "n_edges"),
        "graph_density": safe_get(s, "graph_density"),

        "share_intra_lag": safe_get(e, "share_intra_lag"),
        "share_cross_lag": safe_get(e, "share_cross_lag"),
        "share_same_level": safe_get(e, "share_same_level"),
        "share_adjacent_level": safe_get(e, "share_adjacent_level"),
        "share_distant_level": safe_get(e, "share_distant_level"),
        "share_global_level": safe_get(e, "share_global_level"),
        "share_touches_lag0": safe_get(e, "share_touches_lag0"),

        "mean_abs_precision_weight": safe_get(e, "mean_abs_precision_weight"),
        "median_abs_precision_weight": safe_get(e, "median_abs_precision_weight"),
        "max_abs_precision_weight": safe_get(e, "max_abs_precision_weight"),
        "mean_abs_partial_corr_weight": safe_get(e, "mean_abs_partial_corr_weight"),
        "median_abs_partial_corr_weight": safe_get(e, "median_abs_partial_corr_weight"),
        "max_abs_partial_corr_weight": safe_get(e, "max_abs_partial_corr_weight"),
    }

    rows.append(row)

global_compare = pd.DataFrame(rows)


# ============================================================
# 2) CATEGORY COMPARISON TABLES
# ============================================================

lag_type_compare = pd.concat(
    [
        add_run_col(data["summary_by_lag_type"], run_name)
        for run_name, data in all_data.items()
        if not data["summary_by_lag_type"].empty
    ],
    ignore_index=True,
)

level_relation_compare = pd.concat(
    [
        add_run_col(data["summary_by_level_relation"], run_name)
        for run_name, data in all_data.items()
        if not data["summary_by_level_relation"].empty
    ],
    ignore_index=True,
)

feature_pair_compare = pd.concat(
    [
        add_run_col(data["summary_by_feature_pair"], run_name)
        for run_name, data in all_data.items()
        if not data["summary_by_feature_pair"].empty
    ],
    ignore_index=True,
)

side_pair_compare = pd.concat(
    [
        add_run_col(data["summary_by_side_pair"], run_name)
        for run_name, data in all_data.items()
        if not data["summary_by_side_pair"].empty
    ],
    ignore_index=True,
)


# ============================================================
# 3) PIVOT TABLES FOR EASY COMPARISON
# ============================================================

def pivot_metric(df, index_col, value_col):
    if df.empty or index_col not in df.columns or value_col not in df.columns:
        return pd.DataFrame()

    return (
        df.pivot_table(
            index=index_col,
            columns="run",
            values=value_col,
            aggfunc="first",
        )
        .reset_index()
    )


lag_share_pivot = pivot_metric(lag_type_compare, "lag_relation", "share_edges")
level_share_pivot = pivot_metric(level_relation_compare, "level_relation", "share_edges")
feature_pair_share_pivot = pivot_metric(feature_pair_compare, "feature_pair", "share_edges")
side_pair_share_pivot = pivot_metric(side_pair_compare, "side_pair", "share_edges")


# ============================================================
# 4) TOP EDGE COMPARISON
# ============================================================

top_edges_compare = pd.concat(
    [
        add_run_col(data["top_edges"], run_name)
        for run_name, data in all_data.items()
        if not data["top_edges"].empty
    ],
    ignore_index=True,
)

top_cross_lag_compare = pd.concat(
    [
        add_run_col(data["top_cross_lag"], run_name)
        for run_name, data in all_data.items()
        if not data["top_cross_lag"].empty
    ],
    ignore_index=True,
)

top_intra_lag_compare = pd.concat(
    [
        add_run_col(data["top_intra_lag"], run_name)
        for run_name, data in all_data.items()
        if not data["top_intra_lag"].empty
    ],
    ignore_index=True,
)


# ============================================================
# 5) PRINT REPORT
# ============================================================

print_section("GLASSO STATE GRAPH - LAG COMPARISON REPORT")

print_subsection("1) GLOBAL OVERVIEW")
cols_global = [
    "run",
    "best_alpha",
    "fit_time_sec",
    "n_features",
    "n_edges",
    "graph_density",
    "share_cross_lag",
    "share_same_level",
    "share_adjacent_level",
    "share_global_level",
    "share_touches_lag0",
    "median_abs_precision_weight",
    "max_abs_precision_weight",
]
fmt_df(global_compare[cols_global])

print_subsection("2) FULL GLOBAL METRICS")
fmt_df(global_compare.drop(columns=["base_dir"], errors="ignore"))

print_subsection("3) SHARE BY LAG TYPE")
fmt_df(lag_share_pivot)

print_subsection("4) SHARE BY LEVEL RELATION")
fmt_df(level_share_pivot)

print_subsection("5) SHARE BY SIDE PAIR")
fmt_df(side_pair_share_pivot)

print_subsection("6) TOP FEATURE PAIRS BY RUN")
for run_name in RUNS.keys():
    print()
    print(f"--- {run_name} ---")
    df = feature_pair_compare[feature_pair_compare["run"] == run_name].copy()
    if df.empty:
        print("[empty]")
    else:
        df = df.sort_values("share_edges", ascending=False)
        fmt_df(
            df[
                [
                    "feature_pair",
                    "n_edges",
                    "share_edges",
                    "mean_abs_precision_weight",
                    "median_abs_precision_weight",
                    "max_abs_precision_weight",
                    "mean_abs_partial_corr_weight",
                    "max_abs_partial_corr_weight",
                ]
            ],
            n=TOP_N,
        )

print_subsection("7) TOP OVERALL EDGES BY RUN")
for run_name in RUNS.keys():
    print()
    print(f"--- {run_name} ---")
    df = top_edges_compare[top_edges_compare["run"] == run_name].copy()
    if df.empty:
        print("[empty]")
    else:
        cols = [
            "rank",
            "source",
            "target",
            "abs_precision_weight",
            "partial_corr_weight",
            "sign",
            "lag_relation",
            "level_relation",
            "feature_pair",
            "side_pair",
            "feature_type_pair",
        ]
        cols = [c for c in cols if c in df.columns]
        fmt_df(df[cols], n=TOP_N)

print_subsection("8) TOP CROSS-LAG EDGES BY RUN")
for run_name in RUNS.keys():
    print()
    print(f"--- {run_name} ---")
    df = top_cross_lag_compare[top_cross_lag_compare["run"] == run_name].copy()
    if df.empty:
        print("[empty]")
    else:
        cols = [
            "rank",
            "source",
            "target",
            "abs_precision_weight",
            "partial_corr_weight",
            "lag_relation",
            "level_relation",
            "feature_pair",
            "side_pair",
        ]
        cols = [c for c in cols if c in df.columns]
        fmt_df(df[cols], n=TOP_N)

print_subsection("9) TOP INTRA-LAG EDGES BY RUN")
for run_name in RUNS.keys():
    print()
    print(f"--- {run_name} ---")
    df = top_intra_lag_compare[top_intra_lag_compare["run"] == run_name].copy()
    if df.empty:
        print("[empty]")
    else:
        cols = [
            "rank",
            "source",
            "target",
            "abs_precision_weight",
            "partial_corr_weight",
            "lag_relation",
            "level_relation",
            "feature_pair",
            "side_pair",
        ]
        cols = [c for c in cols if c in df.columns]
        fmt_df(df[cols], n=TOP_N)


# ============================================================
# 6) AUTOMATIC INTERPRETATION
# ============================================================

print_subsection("10) COMPACT AUTOMATIC INTERPRETATION")

best_fit = global_compare.sort_values("fit_time_sec").iloc[0]
most_edges = global_compare.sort_values("n_edges", ascending=False).iloc[0]
highest_cross_lag = global_compare.sort_values("share_cross_lag", ascending=False).iloc[0]
highest_global = global_compare.sort_values("share_global_level", ascending=False).iloc[0]
highest_same_level = global_compare.sort_values("share_same_level", ascending=False).iloc[0]

print(f"- Fastest run              : {best_fit['run']} ({best_fit['fit_time_sec']:.2f} sec)")
print(f"- Most connected graph     : {most_edges['run']} ({int(most_edges['n_edges'])} edges)")
print(f"- Strongest temporal graph : {highest_cross_lag['run']} (cross-lag share={highest_cross_lag['share_cross_lag']:.4f})")
print(f"- Most global-driven graph : {highest_global['run']} (global-level share={highest_global['share_global_level']:.4f})")
print(f"- Most local-level graph   : {highest_same_level['run']} (same-level share={highest_same_level['share_same_level']:.4f})")

print()
print("Suggested reading:")
print("- lag_3  : compact baseline, easier to interpret")
print("- lag_10 : balanced compromise between temporal richness and readability")
print("- lag_20 : long-memory experiment, richer but more autoregressive and heavier")

print()
print_rule("=")
print("END OF COMPARISON REPORT")
print_rule("=")